# Structured Output

Model can be requested to provide their response in a format, matchig a given schema. This is useful for ensuring the output can be easily parsed and  used in subsequent processing. LangChain supports multiple schema types and methods for enforcing structured output

## Pydantic

pydantic models provides the richest feature set with field validation, descriptions, and nested structures

In [3]:
import os
from langchain.chat_models import init_chat_model
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
model = init_chat_model("groq:openai/gpt-oss-20b")

In [4]:
from pydantic import BaseModel, Field
class Movie(BaseModel):
    title:str = Field(description = "The title of the movie")
    year:int = Field(description  =  "This year the movie was released")
    director:str = Field(description= "The director of the movie")
    rating:float = Field(description= "The movies rating out of 10")
    # for field validation we must use string for title, integer for year  etc.

In [5]:
model_with_structure  = model.with_structured_output(Movie)
model_with_structure

_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.7.3', 'langchain': '1.4.7'}}, profile={'name': 'GPT OSS 20B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x00000206B1EF9700>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x00000206B220E6F0>, model_name='openai/gpt-oss-20b', model_kwargs={}, groq_api_key=SecretStr('**********')), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'The titl

In [6]:
model.invoke("Provide details about the movie Inception")  ## this is default response without structured output

AIMessage(content='**Inception** (2010) – A mind‑bending sci‑fi thriller\n\n| Item | Details |\n|------|---------|\n| **Director** | Christopher\u202fNolan |\n| **Screenwriter** | Christopher\u202fNolan |\n| **Producer** | Emma\u202fThomas, Christopher\u202fNolan, Emma\u202fThomas, and others |\n| **Production Company** | Syncopy, Warner Bros. Pictures |\n| **Release Dates** | U.S. (theatrical): 16\u202fJuly\u202f2010 (in 2\u202f,000+ theaters) <br>UK: 17\u202fJuly\u202f2010 |\n| **Runtime** | 2\u202f:28\u202fmin |\n| **Language** | English |\n| **Budget** | ≈\u202f$160\u202fmillion |\n| **Box‑office** | $836\u202fmillion (worldwide) |\n| **Genres** | Science‑fiction, Thriller, Action, Adventure |\n| **Primary Themes** | Dream‑in‑dream architecture, the nature of reality, guilt, memory, grief, the subconscious, time dilation |\n\n---\n\n### Plot (≈\u202f200 words)\n\nDom\u202fCobb (Leonardo DiCaprio) is a “extractor” who enters people’s dreams to steal secrets. After a failed job, he i

In [7]:
#this is structured way 
response  = model_with_structure.invoke("Provide details about the movie Inception") 
response

Movie(title='Inception', year=2010, director='Christopher Nolan', rating=8.8)

#### Message output alongside parsed structure

This shows structured output , also raw output 

In [8]:
from pydantic import BaseModel, Field
class Movie(BaseModel):
    title:str = Field(..., description = "The title of the movie")
    year:int = Field(..., description  =  "This year the movie was released")
    director:str = Field(..., description = "The director of the movie")
    rating:float = Field(..., description = "The movies rating out of 10")
    # for field validation we must use string for title, integer for year  etc.

In [9]:
model_with_structure  = model.with_structured_output(Movie, include_raw=True) # include_raw is responsible here

response  = model_with_structure.invoke("Provide details about the movie Inception") 
response

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': "User wants details about the movie Inception. We have a function to call. We need to call functions.Movie with director, rating, title, year. Provide details. Probably rating 8.8 or 8.8/10. Year 2010. Director Christopher Nolan. Let's call function.", 'tool_calls': [{'id': 'fc_cacac79d-986e-4c58-9b67-8bd05ad6f638', 'function': {'arguments': '{"director":"Christopher Nolan","rating":8.8,"title":"Inception","year":2010}', 'name': 'Movie'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 101, 'prompt_tokens': 158, 'total_tokens': 259, 'completion_time': 0.111029274, 'completion_tokens_details': {'reasoning_tokens': 62}, 'prompt_time': 0.011416918, 'prompt_tokens_details': None, 'queue_time': 0.282426947, 'total_time': 0.122446192}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_8d13edce1d', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model

### Nested Structure

In [9]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name:str
    role:str
    
class MovieDeatils(BaseModel):
    title: str
    year:int
    cast:list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in millions USD")

model_with_structure =  model.with_structured_output(MovieDeatils)
response =  model_with_structure.invoke("Provide details about the movie Inception")
response

MovieDeatils(title='Inception', year=2010, cast=[Actor(name='Leonardo DiCaprio', role='Dom Cobb'), Actor(name='Joseph Gordon-Levitt', role='Arthur'), Actor(name='Elliot Page', role='Ariadne'), Actor(name='Tom Hardy', role='Eames'), Actor(name='Ken Watanabe', role='Saito'), Actor(name='Cillian Murphy', role='Robert Fischer')], genres=['Action', 'Adventure', 'Sci-Fi'], budget=160000000.0)

## TypedDict 

TypeDict provides a simpler alternative using Pythons built in typing , ideal when you dont need runtime validation

In [5]:
from typing_extensions import TypedDict, Annotated

class MovieDict(TypedDict):
    """ A movie with details """
    title:Annotated[str,..., "The title of the movie"]
    year:Annotated[int,... , "The year the movie was released"]
    director: Annotated[str, ..., "The director of the movie"]
    rating:Annotated[float,..., "The movie's rating out of 10"]

In [6]:
model_with_typeddict = model.with_structured_output(MovieDict)
response = model_with_typeddict.invoke("Please provide the details of the movie Avengers")
response

{'director': 'Joss Whedon', 'rating': 8, 'title': 'The Avengers', 'year': 2012}

#### Nested Structure

In [11]:
class Actor(TypedDict):
    name:str
    role:str
    
class MovieDetails(TypedDict):
    title: str
    year:int
    cast:list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in millions USD")

model_with_structure =  model.with_structured_output(MovieDetails)
response =  model_with_structure.invoke("Provide details about the movie Avengers")
response

{'budget': 220000000,
 'cast': [{'name': 'Robert Downey Jr.', 'role': 'Tony Stark / Iron Man'},
  {'name': 'Chris Evans', 'role': 'Steve Rogers / Captain America'},
  {'name': 'Mark Ruffalo', 'role': 'Bruce Banner / Hulk'},
  {'name': 'Chris Hemsworth', 'role': 'Thor'},
  {'name': 'Jeremy Renner', 'role': 'Clint Barton / Hawkeye'},
  {'name': 'Tom Hiddleston', 'role': 'Loki'},
  {'name': 'Samuel L. Jackson', 'role': 'Nick Fury'}],
 'genres': ['Action', 'Adventure', 'Sci-Fi'],
 'title': 'The Avengers',
 'year': 2012}

In [ ]:
model.profile #default model shows profile

{'name': 'GPT OSS 20B',
 'release_date': '2025-08-05',
 'last_updated': '2026-05-27',
 'open_weights': True,
 'max_input_tokens': 131072,
 'max_output_tokens': 65536,
 'text_inputs': True,
 'image_inputs': False,
 'audio_inputs': False,
 'video_inputs': False,
 'text_outputs': True,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True,
 'structured_output': True,
 'attachment': False,
 'temperature': True}

In [ ]:
model_with_typeddict.profile  # structured output model dont show profile

### DataClasses
A data class is a class typically containing mainly data, although there aren't really any restrictions. You create it using the @dataclass decorator 

In [ ]:
## using typedict

import os
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
from pydantic import BaseModel, Field
from langchain.agents import create_agent

class ContactInfo(BaseModel):
    """Contact information for a person."""
    name: str = Field(description="The name of the person")
    email: str = Field(description="The email address of the person")
    phone: str = Field(description="The phone number of the person")

agent = create_agent(
    model="groq:openai/gpt-oss-20b",
    response_format=ContactInfo
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john#example.com, (555) 123-4567"}]
})

print(result["structured_response"])

name='John Doe' email='john#example.com' phone='(555) 123-4567'


In [20]:
## how with the help of Dataclass you can do same thing
from dataclasses import dataclass
from langchain.agents import create_agent

@dataclass
class ContactInfo:
    """Contact information for a person."""
    name: str # The name of the person
    email: str # The email address of the person
    phone: str #the phone number of the person
    
agent = create_agent(
    model="groq:openai/gpt-oss-20b",
    response_format=ContactInfo # auto selects provider strategy
)

result =  agent.invoke({
    "messages":[{"role":"user","content":"Extract contact info from: John Doe, John@example.com, (555) 123-4567"}]
    
})

result["structured_response"]

ContactInfo(name='John Doe', email='John@example.com', phone='(555) 123-4567')